# 面向对象：类与对象

---

**学习目标**

1. 理解"类 = 图纸，对象 = 按图纸造出来的东西"
2. 分清**类属性**和**实例属性**（最容易混的一点）
3. 会用 `__init__` 和 `self`
4. 理解封装（私有属性）和继承、多态
5. 学会用"魔法方法"让自定义对象更好用

**为什么要用类？**

假设要存 3 个学生的信息。用字典可以这么写：

```python
stu1 = {'name': '张三', 'age': 20}
stu2 = {'name': '李四', 'age': 21}
```

问题来了：字段名全靠手打，**打错一个字母不会报错，只会默默存进去**。
而且"学生能做什么事"（比如自我介绍）只能散落在各种函数里。

**类把"数据"和"操作数据的方法"打包在一起**，这才是它的核心价值。


---
## 一、第一个类：属性 + 方法 + `self`


In [ ]:
class Person:
    # 这两行写在类里、方法外面 —— 它们叫【类属性】
    name = None
    age = None

    def say_hello(self):
        # self 就是"当前这个对象本身"，通过它才能访问属于这个对象的数据
        print(f'hello {self.name}')

    def say_hi(self, word):
        # 除了 self，还可以接收别的参数
        print(f'hi {self.name}, {word}')


# 创建对象（实例化）
stu = Person()

# 给这个对象的属性赋值
stu.name = '李子涵'
stu.age = 20

stu.say_hello()
stu.say_hi('我爱学高数')


### `self` 到底是什么？

`self` 就是**调用这个方法的那个对象**。

```python
stu.say_hello()
# 等价于 Person.say_hello(stu)
```

Python 会自动把 `stu` 作为第一个参数传进去，所以方法定义时必须写 `self`。

**理解方式**：把 `stu.say_hello()` 读成"**让 stu 自己执行 say_hello**"，
方法里的 `self` 就是"我自己"。


In [ ]:
# 证明一下：两种调用方式是等价的
stu2 = Person()
stu2.name = '周杰伦'
Person.say_hello(stu2)      # 手动把对象当第一个参数传进去，效果一样


### 一个真实的小例子：会响的闹钟

原始笔记里有个 `clock` 类，用到了 Windows 专有的 `winsound` 模块：


In [ ]:
class Clock:
    """一个闹钟，可以设置编号和价格，然后响铃"""

    id = None
    price = None

    def ring(self):
        import sys
        if sys.platform == 'win32':
            import winsound
            winsound.Beep(2000, 300)      # 频率 2000Hz，持续 300 毫秒
        else:
            # 非 Windows 系统没有 winsound，用打印代替
            print('  （当前系统不支持 winsound，用文字代替铃声：叮铃铃～）')


c1 = Clock()
c1.id = '00031'
c1.price = 100
print(f'闹钟 {c1.id}，价格 {c1.price} 元')
c1.ring()

# > **原始笔记修正说明**
# >
# > 原来直接 `import winsound`，这个模块**只在 Windows 上存在**，
# > 换到 macOS / Linux 就会 `ModuleNotFoundError`。
# > 这里加了 `sys.platform` 判断，让代码跨平台也能跑。
# > 这个小改动体现了一个工程习惯：**不要把平台相关的东西写死在主逻辑里。**


---
## 二、`__init__`：创建对象时自动执行的"装配车间"

上面那种"先创建对象、再一个个赋值"的写法太啰嗦。
`__init__` 允许你在创建对象的那一刻就把数据传进去。


In [ ]:
class Student:
    def __init__(self, name, age, tel):
        # 参数名和属性名同名没关系：self.name 是"对象的属性"，右边的 name 是"传入的参数"
        self.name = name
        self.age = age
        self.tel = tel
        print(f'Student 自动创建了一个对象：{name}')


stu = Student('周杰伦', 31, '1354564654654')
print(f'{stu.name} / {stu.age} / {stu.tel}')


**关于 `__init__` 的几个要点**：

1. 它**不是**构造函数（真正创建对象的是 `__new__`），但它是"初始化方法"
2. 创建对象时**自动调用**，不需要你手动喊它
3. 如果定义了 `__init__` 且要求参数，那创建对象时**必须提供**，否则 `TypeError`
4. `self.name = name` 里，**左边是对象属性，右边是参数** —— 别忘了 `self.`


In [ ]:
try:
    stu_bad = Student('只有名字')
except TypeError as e:
    print('参数没给全 ->', e)


### 实例属性 vs 类属性（**最容易混的一节**）

| | 类属性 | 实例属性 |
|---|---|---|
| 写在哪 | 类里、方法**外** | `__init__` 里用 `self.xxx = ` |
| 属于谁 | **整个类共享一份** | **每个对象各自一份** |
| 读取 | `类名.属性` 或 `对象.属性` | `对象.属性` |
| 修改影响 | 所有**还没自己定义过**该属性的对象都会变 | 只影响自己 |

**经典陷阱：可变类属性被所有实例共享。**


In [ ]:
class Dog:
    species = '犬科'                 # 类属性：所有狗都是犬科
    tricks = []                     # 【危险】可变类属性！

    def __init__(self, name):
        self.name = name            # 实例属性：每个狗的自己的名字

    def add_trick(self, trick):
        self.tricks.append(trick)   # 注意：改的是【类属性】那个列表


d1 = Dog('旺财')
d2 = Dog('来福')

print('两只狗的名字：', d1.name, d2.name)
print('共享的物种：', d1.species, d2.species)

d1.add_trick('坐下')
print('d1 会的：', d1.tricks)
print('d2 会的：', d2.tricks)      # 也有"坐下"！这不是 d2 学的，是共享列表搞的鬼


**看到问题了吗？** `d2` 根本没学过"坐下"，但它也会了。

原因：`tricks` 在**类**上定义，是**一份共享**的列表，两个对象操作的是**同一个列表对象**。

**修正办法**：可变的东西一律放进 `__init__`，让每个对象拥有自己的一份。


In [ ]:
class DogFixed:
    species = '犬科'                 # 不可变的类属性，共享没问题

    def __init__(self, name):
        self.name = name
        self.tricks = []            # 【修正】每个对象自己的列表

    def add_trick(self, trick):
        self.tricks.append(trick)


f1 = DogFixed('旺财')
f2 = DogFixed('来福')
f1.add_trick('坐下')

print('f1 会的：', f1.tricks)
print('f2 会的：', f2.tricks)      # 空的，这才是正确的行为


> **原始笔记的修正说明**
>
> 原笔记里 `class student` 把 `name/age/tel` 都写在类里并赋值为 `None`，
> 随后又在 `__init__` 里重新赋值。这在逻辑上是**冗余**的：
>
> ```python
> class student:
>     name = None       # <- 这一层其实可以省略
>     age = None
>     tel = None
>     def __init__(self, name, age, tel):
>         self.name = name    # __init__ 里会创建【实例属性】，把类属性遮住
>         ...
> ```
>
> 写在类里的那三个 `None` 会被 `__init__` 创建的实例属性"遮住"，
> 所以留着它们除了增加阅读负担没有实际作用（除非你想给一个"默认值"）。
> 一般建议：**想要默认值就在 `__init__` 的参数里给默认值**，比如 `def __init__(self, name='匿名')`。


---
## 三、封装：把"不想让人碰"的东西藏起来

Python 用一个简单的命名约定来实现"私有"：

| 写法 | 名字 | 效果 |
|---|---|---|
| `name` | 公开 | 谁都能访问 |
| `_name` | 单下划线 | **约定**是"内部的，请别碰"，但技术上还能访问 |
| `__name` | 双下划线 | 触发**名称改写**，外部几乎访问不到 |


In [ ]:
class MyClass:
    def __init__(self):
        self.public_var = '公开变量'
        self._protected_var = '约定protected（别在外面用）'
        self.__private_var = '私有变量'

    def get_private_var(self):
        """想读私有变量？通过这个"接口"来读"""
        return self.__private_var


obj = MyClass()
print('公开变量 ->', obj.public_var)
print('单下划线 ->', obj._protected_var)
print('通过接口读私有 ->', obj.get_private_var())


> **原始笔记的修正说明（一个真实 bug）**
>
> 原笔记最后写的是：
>
> ```python
> print(obj.get_private_var)   # 注释写着"输出：私有变量"
> ```
>
> **但实际输出的是一个方法对象**，类似 `<bound method MyClass.get_private_var of ...>`。
> 因为 `get_private_var` 后面**少了一对括号** —— 不带括号表示"这个方法本身"，
> 带上括号才是"调用它、拿到返回值"。
>
> 这是一个非常经典的初学者错误，**务必记住：调用方法一定要加 `()`**。


In [ ]:
# 看看那个"少写括号"的实际输出
print('不带括号 ->', obj.get_private_var)
print('带括号   ->', obj.get_private_var())

# 再试试直接访问私有变量
try:
    print(obj.__private_var)
except AttributeError as e:
    print('直接访问私有变量 ->', e)


### "私有"到底是怎么实现的？——名称改写

Python 并不是真的禁止访问，它只是**把名字改掉了**：
在类内部写 `__private_var`，Python 实际存成了 `_MyClass__private_var`。


In [ ]:
print('对象的真实属性：', [k for k in dir(obj) if 'var' in k])
print('绕过限制硬读 ->', obj._MyClass__private_var)
# 所以准确说法是：Python 的私有是"防手滑"，不是"防黑客"。
# 它的目的是【告诉你别动】，而不是【让你动不了】。


---
## 四、继承：子类自动获得父类的能力

**类比**：父类是"通用手机"，子类是"游戏手机"。
游戏手机**自动拥有**通用手机的全部功能，还可以**额外加**自己的东西，或者**改写**不满意的部分。


In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        """父类先给一个默认实现，子类可以重写它"""
        print(f'{self.name}：……（动物发出了一点声音）')

    def introduce(self):
        print(f'我是一只叫 {self.name} 的动物')


class Dog(Animal):                  # (Animal) 表示"继承自 Animal"
    def speak(self):
        print(f'{self.name}：汪汪汪')


class Cat(Animal):
    def speak(self):
        print(f'{self.name}：喵喵喵')


dog = Dog('旺财')
cat = Cat('咪咪')

dog.introduce()     # 用的是【父类】的方法（继承来的）
dog.speak()         # 用的是【子类】自己重写的方法

cat.introduce()
cat.speak()


### 多态：同一个调用，不同对象有不同表现

下面这个函数只写了 `animal.speak()`，但它能同时处理 `Dog`、`Cat` 甚至未来新加的 `Bird`。
这就是**多态** —— 程序不需要知道具体是什么动物，只要知道"它会 speak"。


In [ ]:
def make_noise(animal: Animal):
    animal.speak()


for a in [Dog('大黄'), Cat('小白'), Dog('黑豆')]:
    make_noise(a)

# 好处：以后加一个新动物，只要它继承 Animal 并实现 speak，
# 这个 make_noise 函数【一行都不用改】。这叫"对扩展开放，对修改关闭"。


### `super()`：调用父类的方法

子类想在父类的基础上"再补充一点"，而不是完全替换，就用 `super()`。


In [ ]:
class Guide(Animal):
    def __init__(self, name, command):
        # 先让父类把 name 处理好（避免重复写 self.name = name）
        super().__init__(name)
        self.command = command

    def speak(self):
        super().speak()                     # 先执行父类的版本
        print(f'  （导盲犬 {self.name} 听到"{self.command}"会站住）')

    def introduce(self):
        print(f'我是一只导盲犬，叫 {self.name}，会听"{self.command}"')


g = Guide('豆豆', '停')
g.introduce()
g.speak()


### 判断对象的类型：`isinstance` 和 `issubclass`


In [ ]:
print('dog 是 Dog 吗？      ', isinstance(dog, Dog))
print('dog 是 Animal 吗？   ', isinstance(dog, Animal))     # True，子类也是父类的一种
print('cat 是 Dog 吗？      ', isinstance(cat, Dog))
print('Dog 是 Animal 的子类吗？', issubclass(Dog, Animal))
print('Animal 是 Dog 的子类吗？', issubclass(Animal, Dog))   # False，方向反了


**重要原则**：`isinstance(dog, Dog)` 和 `isinstance(cat, Animal)` 都为 `True`，
但 `isinstance(cat, Dog)` 为 `False`。继承是**单方向**的。


---
## 五、魔法方法：让自定义对象"更像内置类型"

双下划线开头结尾的方法（`__xxx__`）叫**魔法方法**，
它们让自定义对象能支持 `print`、`len`、`==`、`<` 等操作。


In [ ]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self):
        """str(book) / print(book) 时调用。给【人】看的"""
        return f'《{self.title}》'

    def __repr__(self):
        """在列表里打印、调试时调用。给【开发者】看的"""
        return f'Book({self.title!r}, {self.author!r}, {self.pages})'

    def __len__(self):
        """len(book) 时调用"""
        return self.pages

    def __eq__(self, other):
        """book1 == book2 时调用"""
        return self.title == other.title and self.author == other.author

    def __lt__(self, other):
        """book1 < book2 时调用，有了它就能排序"""
        return self.pages < other.pages


b1 = Book('Python入门', '张三', 320)
b2 = Book('算法导论', '李四', 800)
b3 = Book('Python入门', '张三', 320)

print('print(book)      ->', b1)
print('repr(book)       ->', repr(b2))
print('len(book)        ->', len(b2))
print('两本相同的书 == ? ->', b1 == b3)
print('排序（按页数）    ->', sorted([b2, b1]))
print('列表里打印        ->', [b1, b2])


**`__str__` 和 `__repr__` 的区别**：

- `__str__`：给用户看，可以很友好（`《Python入门》`）
- `__repr__`：给开发者看，尽量能"由它重建对象"（`Book('Python入门', '张三', 320)`）

只定义 `__repr__` 时，`print()` 也会用它。**实在只写一个，就写 `__repr__`。**


---
## 六、实战：把"字典存学生"改造成"类存学生"

原始笔记里用嵌套字典存学生信息。对比一下两种写法的差别。


In [ ]:
# 【旧写法】嵌套字典
students_dict = {}
for i in range(2):
    name = f'学生{i + 1}'
    students_dict[name] = {
        '姓名': '张三' if i == 0 else '李四',
        '年龄': '20' if i == 0 else '21',
        '地址': '武汉' if i == 0 else '北京',
    }
print('字典写法：', students_dict)
# 缺点：字段名靠字符串手打，'年龄' 写成 '年纪' 不会报错，只会悄悄多一个字段


In [ ]:
# 【新写法】用类
class StudentInfo:
    def __init__(self, name, age, address):
        self.name = name
        self.age = age
        self.address = address

    def __repr__(self):
        return f'{self.name}({self.age}岁, {self.address})'

    def to_dict(self):
        """方便转回字典，导出数据时很有用"""
        return {'姓名': self.name, '年龄': self.age, '地址': self.address}


students = [
    StudentInfo('张三', '20', '武汉'),
    StudentInfo('李四', '21', '北京'),
]

print('对象写法：', students)
print('转回字典：')
for s in students:
    print('  ', s.to_dict())


**类的优势**：

1. **字段名有 IDE 提示**，`s.nmae` 会立刻被标红，字典写错只会静默失败
2. **行为和数据在一起**，比如新加一个 `s.introduce()` 方法，调用方直接用
3. **可以做类型校验**，比如在 `__init__` 里检查年龄是不是数字

什么时候**不**用类？数据只是简单传递、不需要行为时，用字典或 `dataclass` 更省事。


In [ ]:
# 数据校验的例子：让类帮你挡住错误数据
class SafeStudent:
    def __init__(self, name, age):
        self.name = name
        if not isinstance(age, int):
            raise TypeError(f'年龄必须是整数，收到的是 {type(age).__name__}')
        if not 0 < age < 150:
            raise ValueError(f'年龄超出合理范围：{age}')
        self.age = age

    def __repr__(self):
        return f'SafeStudent({self.name!r}, {self.age})'


print(SafeStudent('张三', 20))
for bad in [('李四', '20'), ('王五', 200)]:
    try:
        SafeStudent(*bad)
    except (TypeError, ValueError) as e:
        print('拦住了一个错误数据 ->', e)


---
## 七、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 调用方法忘了加 `()` | 打印出 `<bound method ...>` | 加上括号 `obj.method()` |
| 方法定义忘了写 `self` | `TypeError: takes 0 positional arguments but 1 was given` | 第一个参数写 `self` |
| 访问属性忘了写 `self.` | `NameError` | 方法里访问自己的属性必须 `self.xxx` |
| 可变类属性被共享 | 一个实例改了，所有实例都变 | 可变数据放进 `__init__` |
| `__init__` 里参数名和属性名搞混 | 属性拿到了错的参数 | 记住「左 self，右参数」 |
| 以为 `__xxx` 真的私有 | `_类名__xxx` 能绕过 | 它是"防手滑"，不是安全机制 |
| 继承时忘了 `super().__init__()` | 父类属性没初始化 | 需要父类初始化就调用它 |
| 靠继承来"复用几个方法" | 继承层次混乱，改一处崩一片 | 优先考虑组合（把一个对象当属性） |
| 忘了 `__repr__` | 打印出来是 `<object at 0x...>` | 加上 `__repr__` 方便调试 |

> 原始笔记开头还有一行 `from pandas.core.interchange.from_dataframe import primitive_column_to_ndarray`，
> 这和"类"毫无关系，属于误粘贴的导入，**已删除**。
> 这类无关导入会让人误以为"学类还需要 pandas"，实际完全不需要。


---
## 八、动手练习

**练习 1**：写一个 `Rectangle` 类，有 `width`、`height` 两个属性，
方法 `area()` 返回面积、`perimeter()` 返回周长。

**练习 2**：给 `Rectangle` 加上 `__repr__` 和 `__eq__`（长宽都相同视为相等）。

**练习 3**：写一个 `BankAccount` 类：
- `__init__(self, owner, balance=0)`
- `deposit(amount)` 存款，`withdraw(amount)` 取款
- 余额不足时**抛异常**而不是打印提示
- `__repr__` 显示 `户主(余额)`

**练习 4**：继承练习 3 的 `BankAccount`，写一个 `SavingsAccount`（储蓄账户），
每次存款额外给 1% 利息（用 `super()` 复用父类的存款逻辑）。

**练习 5**：写一个 `Playlist` 类，用 `__len__` 返回歌曲数、`__getitem__` 支持 `playlist[0]`，
让它能像列表一样用 `for` 遍历。

**练习 6**：解释下面代码为什么报错，并改对：
```python
class A:
    def __init__(self, x):
        self.x = x
    def show():
        print(x)
```

---

**答案思路（先自己做）**

```python
# 练习 3
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('存款金额必须为正数')
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f'余额不足：当前 {self.balance}，尝试取 {amount}')
        self.balance -= amount
        return self.balance

    def __repr__(self):
        return f'{self.owner}({self.balance})'


# 练习 4
class SavingsAccount(BankAccount):
    def deposit(self, amount):
        super().deposit(amount)        # 先走父类逻辑
        self.balance *= 1.01           # 再额外加 1% 利息
        return self.balance
```
